In [6]:
import os, sys, glob, shutil

def find_first(tail, depths=range(1, 9)):
    for d in depths:
        hits = sorted(glob.glob("/kaggle/input/" + "*/" * d + tail))
        if hits:
            return hits[0]
    return None

# Repo: find the folder containing src/, copy it into writable /kaggle/working
cfg = find_first("src/config.py")
assert cfg, "Repo not found — add your repo zip dataset."
repo_src = os.path.dirname(os.path.dirname(cfg))
REPO = "/kaggle/working/" + os.path.basename(repo_src.rstrip("/"))
if os.path.exists(REPO):
    shutil.rmtree(REPO)
shutil.copytree(repo_src, REPO)
sys.path.insert(0, REPO)

# Model lives OUTSIDE the repo so re-running this cell never deletes it
MODEL_DIR = "/kaggle/working/models"

# Fake-or-Real (for-norm) splits
FOR_TRAIN = find_first("for-norm/training")
assert FOR_TRAIN, "FoR for-norm/training not found."
FOR_DIR  = os.path.dirname(FOR_TRAIN)
FOR_VAL  = os.path.join(FOR_DIR, "validation")
FOR_TEST = os.path.join(FOR_DIR, "testing")

# ASVspoof 2019 LA train
ASV_FLAC  = find_first("ASVspoof2019_LA_train/flac")
ASV_PROTO = find_first("ASVspoof2019.LA.cm.train.trn.txt")
assert ASV_FLAC and ASV_PROTO, "ASVspoof LA train flac/protocol not found."

for k in ["REPO", "MODEL_DIR", "FOR_TRAIN", "FOR_VAL", "FOR_TEST", "ASV_FLAC", "ASV_PROTO"]:
    print(f"{k:9}:", eval(k))

REPO     : /kaggle/working/deepfake-audio-detection-main
MODEL_DIR: /kaggle/working/models
FOR_TRAIN: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/training
FOR_VAL  : /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/validation
FOR_TEST : /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/testing
ASV_FLAC : /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/flac
ASV_PROTO: /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt


In [7]:
import os
from collections import Counter

def ok(b): return "✅" if b else "❌"

print("=== paths ===")
for k in ["REPO", "FOR_TRAIN", "FOR_VAL", "FOR_TEST", "ASV_FLAC", "ASV_PROTO"]:
    v = eval(k); print(ok(os.path.exists(v)), k, "->", v)

print("\n=== augmentation edits present? (both must be ✅) ===")
feat = open(f"{REPO}/src/features.py").read()
ds   = open(f"{REPO}/src/dataset.py").read()
print(ok("def augment_waveform" in feat), "features.py defines augment_waveform")
print(ok("augment_waveform" in ds),       "dataset.py uses augment_waveform")

print("\n=== ASVspoof LA contents ===")
LA = os.path.dirname(os.path.dirname(ASV_FLAC))
for name in sorted(os.listdir(LA)): print("   ", name)
print("flac files in LA_train:", sum(1 for f in os.listdir(ASV_FLAC) if f.endswith(".flac")))
with open(ASV_PROTO) as fh:
    keys = Counter(line.split()[-1].lower() for line in fh if len(line.split()) >= 2)
print("protocol labels:", dict(keys))

=== paths ===
✅ REPO -> /kaggle/working/deepfake-audio-detection-main
✅ FOR_TRAIN -> /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/training
✅ FOR_VAL -> /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/validation
✅ FOR_TEST -> /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/testing
✅ ASV_FLAC -> /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/flac
✅ ASV_PROTO -> /kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt

=== augmentation edits present? (both must be ✅) ===
✅ features.py defines augment_waveform
✅ dataset.py uses augment_waveform

=== ASVspoof LA contents ===
    ASVspoof2019_LA_asv_protocols
    ASVspoof2019_LA_asv_scores
    ASVspoof2019_LA_cm_protocols
    ASVspoof2019_LA_dev
    ASVspoof2019_LA_eval
    ASVspoof2019_LA_train
    README.LA.txt
flac files in LA_train

In [8]:
!pip install -q librosa soundfile pyyaml
!nvidia-smi

Mon Jun 15 04:17:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!python {REPO}/scripts/prepare_dataset.py --root {FOR_DIR}

Scanning splits under: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm

  testing        total=4634   Genuine=2264   Deepfake=2370  
  training       total=53868  Genuine=26941  Deepfake=26927 
  validation     total=10798  Genuine=5400   Deepfake=5398  

Label convention: 0 = Genuine (Human), 1 = Deepfake (AI-Generated).
If the counts look right, you're ready to train:  python -m src.train


In [10]:
import os
from pathlib import Path
from collections import Counter
from src.dataset import scan_dataset

MERGED = "/kaggle/working/merged_train"
Path(MERGED, "real").mkdir(parents=True, exist_ok=True)
Path(MERGED, "fake").mkdir(parents=True, exist_ok=True)

def link(src, label):
    dst = Path(MERGED, "real" if label == 0 else "fake", Path(src).name)
    if not dst.exists():
        try: os.symlink(src, dst)
        except (FileExistsError, OSError): pass

n_for = 0
for p, lbl in scan_dataset(FOR_TRAIN):
    link(p, lbl); n_for += 1
print("FoR training linked:", n_for)

n_b = n_s = n_miss = 0
with open(ASV_PROTO) as fh:
    for line in fh:
        parts = line.split()
        if len(parts) < 2: continue
        fname, key = parts[1], parts[-1].lower()
        flac = os.path.join(ASV_FLAC, fname + ".flac")
        if not os.path.exists(flac): n_miss += 1; continue
        if key == "bonafide": link(flac, 0); n_b += 1
        elif key == "spoof":  link(flac, 1); n_s += 1
print(f"ASVspoof linked: bonafide(real)={n_b} spoof(fake)={n_s} missing={n_miss}")

c = Counter(l for _, l in scan_dataset(MERGED))
print("MERGED totals -> genuine(0):", c.get(0, 0), " deepfake(1):", c.get(1, 0))

FoR training linked: 53868
ASVspoof linked: bonafide(real)=2580 spoof(fake)=22800 missing=0
MERGED totals -> genuine(0): 29521  deepfake(1): 49727


In [18]:
%cd {REPO}
!python -u -m src.train \
  --train-dir {MERGED} \
  --val-dir {FOR_VAL} \
  --model-dir {MODEL_DIR} \
  --epochs 12  --batch-size 64 --num-workers 4

/kaggle/working/deepfake-audio-detection-main
[train] device=cuda  amp=True
[train] train samples: {1: 49727, 0: 29521} (total 79248)
[train] val   samples: {1: 5398, 0: 5400} (total 10798)
[train] model=specnet_cnn  params=590,690
[train] class weights: [1.2549717426300049, 0.7450282573699951]
/kaggle/working/deepfake-audio-detection-main/src/train.py:189: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/kaggle/working/deepfake-audio-detection-main/src/train.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[epoch 01/12] train_loss=0.5121 acc=0.758 | val_loss=0.2942 acc=0.885 EER=10.61% | lr=1.00e-03
          ^ new best (val EER=10.61%, acc=0.885)
/kaggle/working/deepfake-audio-detection-main/src/train.py:73: FutureWarning: `torch.cuda.amp.a

In [19]:
!python -m src.evaluate \
  --test-dir {FOR_TEST} \
  --model {MODEL_DIR}/best_model.pt

[eval] test samples: {1: 2370, 0: 2264} (total 4634)

  DEEPFAKE AUDIO DETECTION — EVALUATION REPORT
  Samples evaluated : 4634
  Genuine / Deepfake: 2264 / 2370
--------------------------------------------------------
  PRIMARY METRICS
    Overall Accuracy :  92.27%   (require >= 80%)  [PASS]
    Equal Error Rate :   5.53%   (require <= 12%)  [PASS]   @ thr=0.209
--------------------------------------------------------
  SECONDARY METRICS
    F1 (macro)       :  92.26%   (require >= 80%)  [PASS]
      F1 Genuine     :  92.56%
      F1 Deepfake    :  91.97%
    Per-class accuracy (require each >= 75%):
      Genuine        :  98.32%   [PASS]
      Deepfake       :  86.50%   [PASS]
--------------------------------------------------------
  CONFUSION MATRIX  (rows = true, cols = predicted)
                 pred:Genuine   pred:Deepfake
    true:Genuine         2226              38
    true:Deepfake         320            2050
  VERIFICATION (§5): PASS — all thresholds met

[eval] figures 

In [20]:
!mkdir -p /kaggle/working/deliverables && cp {MODEL_DIR}/best_model.pt /kaggle/working/deliverables/ 2>/dev/null
!cp -r {REPO}/reports /kaggle/working/deliverables/reports
!cd /kaggle/working && zip -rq deliverables.zip deliverables && echo "ready: /kaggle/working/deliverables.zip"

ready: /kaggle/working/deliverables.zip
